# Step-Free London — Session 1: building the accessibility dataset

**Question:** where does London's tube network leave wheelchair users behind?

What we discovered while getting the data:

1. The **TfL Unified API** gives us all 272 tube stations with coordinates and lines, but its step-free field (`AccessViaLift`) is only filled for ~30% of stations — even King's Cross St Pancras has *no* accessibility data in the API.
2. **OpenStreetMap** has `wheelchair` tags on 264 of those same 272 stations.
3. Where both sources speak, they disagree on ~31% of stations — and spot-checking shows OSM is *more current* (e.g. Epping, High Barnet and Woodford got lifts years ago; TfL's API still says no).

Final status uses OSM as primary, TfL API as a cross-check.

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/processed/tube_stepfree_final.csv")
print(f"stations: {len(df)}")
df["step_free"].value_counts()


stations: 272


step_free
not step-free    155
step-free         83
partial           26
unknown            8
Name: count, dtype: int64

## How accessible is the network?

Only around 30% of stations are fully step-free; 'partial' means step-free to
some platforms only (like Borough, where you can only exit northbound).

In [2]:
order = ["step-free", "partial", "not step-free", "unknown"]
counts = df["step_free"].value_counts().reindex(order)
fig = px.bar(
    x=counts.index,
    y=counts.values,
    color=counts.index,
    color_discrete_map={
        "step-free": "#2e7d32",
        "partial": "#f9a825",
        "not step-free": "#c62828",
        "unknown": "#9e9e9e",
    },
    labels={"x": "accessibility status", "y": "stations"},
    title="London Underground station accessibility",
)
fig.show()


## Official data vs crowd-sourced data

Every red bar below is a station where the official API is out of date.
This matters for disabled travellers relying on TfL's own tools.

In [3]:
both = df[df["access_via_lift"].notna() & df["wheelchair"].isin(["yes", "no"])].copy()
both["api_cat"] = both["access_via_lift"].str.lower()
agree = (both["api_cat"] == both["wheelchair"]).sum()
print(f"agreement: {agree}/{len(both)} ({agree / len(both):.0%})\n")

disagreed = both[both["api_cat"] != both["wheelchair"]]
disagreed[["name", "lines", "access_via_lift", "wheelchair"]].style.hide(axis="index")


agreement: 43/62 (69%)



name,lines,access_via_lift,wheelchair
Debden Underground Station,Central,No,yes
Dagenham Heathway Underground Station,District,No,yes
Elephant & Castle Underground Station,"Bakerloo, Northern",Yes,no
Epping Underground Station,Central,No,yes
Elm Park Underground Station,District,No,yes
Finsbury Park Underground Station,"Piccadilly, Victoria",No,yes
High Barnet Underground Station,Northern,No,yes
Hammersmith (H&C Line) Underground Station,"Circle, Hammersmith & City",No,yes
Hammersmith (Dist&Picc Line) Underground Station,"District, Piccadilly",No,yes
Paddington Underground Station,"Bakerloo, Circle, District",No,yes


## The map

Green = step-free, amber = partial, red = not step-free, grey = unknown.

In [4]:
%run ../scripts/make_map.py

from IPython.display import IFrame
IFrame("../reports/step_free_london_map.html", width="100%", height=520)


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/tube_stepfree_final.csv'

## Next sessions

- [ ] Fill the 8 unknowns from TfL's official step-free guide PDF (Stratford, West Ham, Canning Town... are all actually step-free!)
- [ ] Add Census 2021 disability prevalence per borough (ONS)
- [ ] Compute the accessibility gap: disabled population vs nearest step-free station distance
- [ ] Per-line analysis: which lines trap their users?